In [ ]:
import gc
import tqdm
import pandas as pd
import xgboost as xgb
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
from sklearn.model_selection import RepeatedKFold
import default_risk.config as cfg
import os
import xgboost as xgb
import numpy as np
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score

import dtale
import mlflow
import mlflow.xgboost
import default_risk.config
from default_risk.scripts.auxiliars_for_modeling import cast_object_into_categoricals
from default_risk.scripts.auxiliars_for_modeling import get_baseline_setup
from default_risk.scripts.auxiliars_for_modeling import prepare_columns
from default_risk.scripts.feature_cleaner import clean_importance_zero_and_negative_pfi
from default_risk.scripts.feature_cleaner import clean_noise_from_feature_importance
from default_risk.scripts.feature_cleaner import creating_criteria

# Mostrar TODAS las filas del DataFrame
pd.set_option('display.max_rows', None)

# Mostrar TODAS las columnas (crucial para tus 360+ features)
pd.set_option('display.max_columns', None)



# Ajustar el ancho de la pantalla para que no se rompa la tabla en la consola
pd.set_option('display.width', 1000)


load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
cv,hiperparams = get_baseline_setup()
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)



In [ ]:


application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_with_kui.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments_time_window.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()



X,Y = prepare_columns(merged_df)
X = cast_object_into_categoricals(X)

#feature_raper= pd.read_csv(cfg.ARTIFACTS_DIR / "final_importance.csv")

#X= clean_noise_from_feature_importance(feature_raper,X)

merged_df= merged_df.drop(columns=["flag_email"])
import logging
logging.getLogger("mlflow").setLevel(logging.ERROR)
logging.getLogger("mlflow.tracking._tracking_service.client").setLevel(logging.ERROR)
indice_inicio = X.columns.get_loc("education_type_Lower secondary")

baseline_oof_auc, baseline_std = run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"complete_baseline")
results = []
features_drop_file = cfg.ARTIFACTS_DIR / 'feature-auc-impact.csv'
#pd.DataFrame(columns=['feature_dropped', 'auc_impact', 'std_impact']).to_csv(features_drop_file, index=False, encoding='utf-8')

columnas_restantes = X.columns[indice_inicio:]

from contextlib import redirect_stderr, redirect_stdout

from tqdm.auto import tqdm
for col in tqdm(columnas_restantes, desc="Evaluating model without variables"):
        X_dropped = X.drop(columns=[col])
        run_name = f"complete_baseline_drop_{col.replace('/', '_')}"
        with open(os.devnull, 'w') as f, redirect_stdout(f), redirect_stderr(f):
            oof_auc, std = run_cv_tracked_mlflow(
                xgb.XGBClassifier, hiperparams, cv, X_dropped, Y, experiment_name, run_name=run_name
            )
        auc_drop = baseline_oof_auc - oof_auc
        std_diff = baseline_std - std
        results.append({
                    'feature_dropped': col,
                    'auc_impact': auc_drop,
                    'std_impact': std_diff
                })
        
        row_df = pd.DataFrame([{
        'feature_dropped': col,
        'auc_impact': auc_drop,
        'std_impact': std_diff
    }])
    
        row_df.to_csv(features_drop_file, mode='a', header=False, index=False, encoding='utf-8')
        print(f"Feature {col} dropped. Result: {auc_drop}, {std_diff}")



#freeing memory
del merged_df
gc.collect()